# 01 · Data Profiling & Quality

This notebook profiles the **pre-cleaned** benchmark dataset before analysis.
Cleaning (outlier removal + unit conversion) was already performed by
`notebooks/01_data_cleaning.ipynb`, which produced `results/results_clean_runs.csv`.

**Cleaning pipeline (upstream):**
- Outliers removed per **(language × benchmark)** group using the 1.5×IQR boxplot fence,
  applied to both CPU energy and execution time
- Units converted: µJ → J, µs → s, µg → g, Bytes → MB, mW → W

**This notebook covers:**
- Data coverage (languages × benchmarks)
- Per-language summary table (mean per metric)

**Dataset:** 18 languages × 8 benchmarks, measured with the Green Metrics Tool (GMT).
**Priority metrics:** CPU Energy (J), Memory Energy (J), Execution Time (s)

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make the shared style module importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations

import importlib
import plot_style as ps
importlib.reload(ps)   # pick up edits to plot_style.py without a kernel restart
ps.apply_style()

# Canonical constants — single source: plot_style.
COL_CPU_ENERGY, COL_MEM_ENERGY = ps.COL_CPU_ENERGY, ps.COL_MEM_ENERGY
COL_TIME                       = ps.COL_TIME
COL_CPU_CARBON, COL_MEM_CARBON = ps.COL_CPU_CARBON, ps.COL_MEM_CARBON
COMPILER        = ps.COMPILER
COMPILER_COLORS = ps.COMPILER_COLORS
COMPILER_ORDER  = ps.COMPILER_ORDER
MEANPROPS       = ps.MEANPROPS
ALPHA           = ps.ALPHA

OUTPUTS_DIR = Path('outputs'); OUTPUTS_DIR.mkdir(exist_ok=True)

# Single source of truth: per-run rows (df) + per-cell means with EDP (df_mean).
df      = ps.load_runs()
df_mean = ps.cell_means(df)

def lang_means(cols):
    """Per-language two-step mean (equal benchmark weight) for column(s) `cols`."""
    return ps.lang_means(df_mean, cols)

print(f"Runs: {df.shape} | Cell-means: {df_mean.shape} | "
      f"{df['language'].nunique()} languages \u00d7 {df['benchmark'].nunique()} benchmarks")
df_mean.head(3)

## 1. Data Coverage

Heatmap showing the number of runs for each language × benchmark pair.
A uniform count across all cells indicates balanced coverage.

In [ ]:
coverage = df.groupby(['language', 'benchmark']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(coverage, annot=True, fmt='d', cmap='Blues', ax=ax,
            linewidths=0.4, linecolor='#ccc',
            cbar_kws={'label': 'Number of runs'})
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
ps.save_fig(fig, '01_coverage_heatmap')
plt.show()
print(f"Min runs in any cell: {coverage.values.min()}")
print(f"Max runs in any cell: {coverage.values.max()}")
print(f"Mean runs per cell:   {coverage.values.mean():.1f}")

> **Takeaway:** every (language × benchmark) cell keeps 5–10 clean runs after outlier removal — coverage is complete and roughly balanced.

## 2. Summary Table

Per-language **mean** for all three priority metrics in human-readable units, ordered by
run count. The median is still computed to flag skewness (mean/median divergence >20% is
printed above), but only the means are shown in the table.

In [ ]:
# Per-language descriptive stats feeding the summary table below. Uses the two-step
# mean (per-(language x benchmark) cell mean -> per-language mean) so every benchmark
# carries equal weight regardless of how many runs survived outlier removal; a plain
# mean over runs would over-weight the cells that kept more runs. The median is the
# median of the same cell means and is used only to flag skewness (divergence >20%).
_stats = df_mean.groupby('language')[[COL_CPU_ENERGY, COL_MEM_ENERGY, COL_TIME]]
summary_tbl = pd.concat([
    _stats.mean().rename(columns={COL_CPU_ENERGY: 'cpu_mean_J',
                                  COL_MEM_ENERGY: 'mem_mean_J',
                                  COL_TIME:       'time_mean_s'}),
    _stats.median().rename(columns={COL_CPU_ENERGY: 'cpu_median_J',
                                    COL_MEM_ENERGY: 'mem_median_J',
                                    COL_TIME:       'time_median_s'}),
], axis=1)
# Clean-run count per language: sampling context only, it no longer weights the means.
summary_tbl.insert(0, 'runs', df.groupby('language')['run_id'].count())
summary_tbl.insert(0, 'compiler', df_mean.groupby('language')['compiler'].first())
summary_tbl = summary_tbl.round(4)

# Flag skew
for label, mc, mdc in [('CPU energy', 'cpu_mean_J', 'cpu_median_J'),
                        ('Mem energy', 'mem_mean_J', 'mem_median_J'),
                        ('Time',       'time_mean_s','time_median_s')]:
    skew_mask = (abs(summary_tbl[mc] - summary_tbl[mdc]) / summary_tbl[mdc]) > 0.20
    if skew_mask.any():
        print(f"⚠ {label} mean/median diverge >20% for: {list(summary_tbl.index[skew_mask])}")

summary_tbl.sort_values('runs', ascending=False)

In [ ]:
# Per-language summary as a paste-ready table figure, ordered by run count (most → least).
# Means only (medians dropped from the export/figure); the skew check above still uses them.
# Values are two-step means (equal benchmark weight), matching the ranking tables in the
# 02/04 notebooks; 'Runs' is the clean-run count and does not weight the means.
MEAN_COLS = ['compiler', 'runs', 'cpu_mean_J', 'mem_mean_J', 'time_mean_s']
summary_runs = summary_tbl.sort_values('runs', ascending=False)[MEAN_COLS]
summary_runs.to_csv(OUTPUTS_DIR / 'per_language_summary.csv')
print(f"Saved → {OUTPUTS_DIR / 'per_language_summary.csv'}")

# Display-formatted copy (strings) for the styled table figure (PNG + PDF).
disp = pd.DataFrame({
    'Execution Model': summary_runs['compiler'],
    'Runs':            summary_runs['runs'].astype(int).astype(str),
    'CPU mean (J)':    summary_runs['cpu_mean_J'].map(lambda v: f'{v:,.2f}'),
    'Mem mean (J)':    summary_runs['mem_mean_J'].map(lambda v: f'{v:,.2f}'),
    'Time mean (s)':   summary_runs['time_mean_s'].map(lambda v: f'{v:,.2f}'),
}, index=summary_runs.index)
disp.index.name = 'Language'

ps.styled_table_fig(
    disp,
    'Per-Language Summary — two-step mean per metric (equal benchmark weight)',
    '01_per_language_summary',
    row_compilers=[COMPILER[l] for l in disp.index],
)
disp